# 01A — Acquisition  ·  line A (NOAA)

Find each active region on the disk, download a tracked JSOC cutout of it in three HMI
series, and stack the frames into cubes.

**The only notebook that needs the network.** Everything downstream works from what it
leaves in `data/raw/`.

| out | `data/raw/NOAA_<noaa>_<date>/region_01/*.fits` — one file per frame per series |
|---|---|
|     | `data/raw/NOAA_<noaa>_<date>/region_01_{continuum,magnetogram,dopplergram}_cube.fits` |

Which regions, which series, and the per-AR overrides all live in `src/config.py`
(`ACTIVE_REGIONS`, `SERIES_LIST`, `SERIES_OVERRIDE`, `SAMPLE_OVERRIDE`, `BOX_OVERRIDE`) —
they used to be retyped in these cells.

### The reference regions

The ARs are the ones Löptien et al. measured Wilson depressions for with Hinode, and which
SDO also observed. That overlap is the whole point: an independent depression measurement to
compare a fitted Doppler amplitude against.

| NOAA | date | θ (°) | area (Mm²) | B_av (G) | z_div (km) | z_press (km) |
|---|---|---|---|---|---|---|
| 11039 | 2010-01-01 | 29 | 284 | 2471 | 659 | 366 |
| 11041 | 2010-01-26 | 20 | 170 | 2068 | 638 | 305 |
| 11106 | 2010-09-16 | 27 | 195 | 2349 | 636 | 381 |
| 11117 | 2010-10-27 | 25 | 522 | 2093 | 561 | 291 |
| 11117 | 2010-10-28 | 35 | 223 | 2090 | 625 | 313 |
| 11363 | 2011-12-06 | 25 | 1268 | 2287 | 524 | 326 |
| 11536 | 2012-07-31 | 34 | 124 | 2181 | 577 | 346 |
| 13131 | 2022-10-29 | — | — | — | 800 | 800 |

11039 and 11041 predate HMI science data (~2010-05-01) and so have no cubes. 13131 is not a
Löptien region: its 800 km is the Romero (2020) value, carried so the region can join the
same scatter plot in `04`.

The same table is `config.LOPTIEN_TABLE`, which is what `04` actually reads.

In [ ]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import astropy.units as u
import matplotlib.pyplot as plt
from astropy.io import fits

from src import config, download

active_regions = [dict(ar) for ar in config.ACTIVE_REGIONS]
print(f'{len(active_regions)} active region(s) to fetch:')
for ar in active_regions:
    print(f"  NOAA {ar['noaa']} {ar['date']}  series={config.series_for(ar['noaa'])}"
          f"{'  sample=' + str(config.sample_for(ar['noaa'])) if config.sample_for(ar['noaa']) else ''}")

## Locating each AR

`locate_ar_window` queries the HEK and returns a **square** box sized from the group's
reported extent across the window, positioned on the anchor day's own centroid. Position is
then left entirely to JSOC's rotation tracking.

Three things learned the hard way from the records HEK actually returns:

- **Provider matters more than statistics.** One NOAA number returns rows from several
  `frm_name` providers describing different things. `NOAA SWPC Observer` is the *sunspot
  group* (half-width ~70–140″); `HMI SHARP` is the whole active-region complex including
  plage (~250–400″). Mixing them is how NOAA 11363 once ended up with a 736″ box around a
  ~110″ spot.
- **The box is square.** SWPC reports an essentially degenerate latitude extent, so its
  height carries no information, and a square box gives the quiet-sun plane fit an isotropic
  footprint.
- **Median, then clamp.** The bounding boxes are coarse and one bad row should not size the
  whole window.

Oversizing is not free — a large box spans a large line-of-sight rotation gradient, which
contaminates any quiet-sun reference computed over it. Undersizing is worse, because it
silently truncates the thing being measured: NOAA 11536's group grows to ~220″ and sits ~35″
off the reported centroid, so it needs `statistic='max'` and extra padding. That override is
in `config.BOX_OVERRIDE` — always check the resulting box against the frames, not on faith.

In [ ]:
for ar in active_regions:
    ar['bl'], ar['tr'], ar['time_start'], ar['time_end'] = download.locate_ar_window(
        ar['noaa'], ar['date'], n_days=config.N_DAYS,
        **config.BOX_OVERRIDE.get(ar['noaa'], {}))
    print(f"  window {ar['time_start']} .. {ar['time_end']}  "
          f"bl={ar['bl'].Tx:.1f},{ar['bl'].Ty:.1f}  tr={ar['tr'].Tx:.1f},{ar['tr'].Ty:.1f}")

## Downloading

`download_regions` skips a region/series already covered on disk and **resumes** one that is
partly covered, so re-running this is cheap and is the intended way to recover from a
failure.

Read the status carefully: `partial` matters as much as `failed`. A missing *mid-window*
frame is not a smaller dataset — it knocks that series out of step with the other two from
that point on. (`02A` puts everything on a uniform grid and leaves such holes as NaN, so
they stay visible rather than silently shifting the series, but the frames are still gone.)

NOAA 11117 is special-cased in `config`: JSOC's 720 s series 500-errors across its window, so
it comes from the 45 s series downsampled server-side to 720 s.

In [ ]:
NOTIFY_EMAIL = config.NOTIFY_EMAIL   # must be a JSOC-registered export email

all_summaries = []
for ar in active_regions:
    region_base = config.RAW_DIR / f"NOAA_{ar['noaa']}_{ar['date']}"
    regions_hpc = [(ar['bl'], ar['tr'])]
    print(f"\nNOAA {ar['noaa']} {ar['date']} -> {region_base}")
    for series in config.series_for(ar['noaa']):
        summary = download.download_regions(
            regions_hpc, region_base, ar['time_start'], ar['time_end'],
            NOTIFY_EMAIL, series, sample=config.sample_for(ar['noaa']))
        all_summaries.append((ar['noaa'], ar['date'], series, summary))

## Did anything go missing?

Re-run the download cell to resume. Only clear a region and start over if the *box* changed.

In [ ]:
by_status = {}
for noaa, date, series, summary in all_summaries:
    for region, info in summary.items():
        by_status.setdefault(info['status'], []).append((noaa, date, series, region, info))

for status in ('failed', 'partial'):
    entries = by_status.get(status, [])
    if entries:
        print(f'\n{status.upper()} downloads:')
        for noaa, date, series, region, info in entries:
            detail = info.get('error') or f"{len(info.get('failed_urls', []))} file(s) missing"
            print(f'  NOAA {noaa} ({date}) region_{region:02d} [{series}]: {detail}')

if all_summaries and not (by_status.get('failed') or by_status.get('partial')):
    print(f'\nAll {len(all_summaries)} region/series downloads complete, no missing files.')

## Preview a frame

A quick look at what actually came down, before spending time on cubes.

In [ ]:
for ar in active_regions:
    region_dir = config.RAW_DIR / f"NOAA_{ar['noaa']}_{ar['date']}" / 'region_01'
    frames = sorted(region_dir.glob('*.continuum.fits'))
    if not frames:
        print(f"NOAA {ar['noaa']}: no continuum frames on disk")
        continue
    with fits.open(frames[0]) as hdul:
        image = hdul[1].data
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image, origin='lower', cmap='gray')
    ax.set_title(f"NOAA {ar['noaa']} — {frames[0].name}\n{image.shape}, {len(frames)} frames")
    plt.tight_layout()
    plt.show()

## Build the cubes

One cube per region per series, with a `TIMESTAMPS` extension so everything downstream
aligns **by time** rather than by frame index.

These are *tracked* cutouts: JSOC has already registered every frame so the region sits at
the same pixel throughout, which is why the frames are stacked by pixel index and must
**not** be reprojected onto a common sky WCS — doing that would undo exactly the alignment
tracking provided.

A fixed-arcsec tracked cutout has a constant pixel shape, so a shape warning here means
frames left over from an earlier download under a different box. Re-download rather than
letting `make_cube` pad them.

In [ ]:
for ar in active_regions:
    region_base = config.RAW_DIR / f"NOAA_{ar['noaa']}_{ar['date']}"
    regions_hpc = [(ar['bl'], ar['tr'])]
    for series in config.series_for(ar['noaa']):
        download.make_cubes(regions_hpc, region_base, series)

---

## Aside — the SHARP keyword query for NOAA 13131

A one-off: 13131 is not in the Löptien table, so its HARP number and area had to be looked up
directly. Kept in the notebook rather than in `src/` precisely because it is called once, to
produce one file (`data/raw/NOAA_13131_2022-10-29/sharp_keys.csv`).

In [ ]:
import drms
import pandas as pd

RUN_SHARP_QUERY = False   # network; the answer is already saved

if RUN_SHARP_QUERY:
    keys = drms.Client().query(
        'hmi.sharp_720s[][2022.10.29_06:12_TAI-2022.10.29_07:12_TAI]',
        key=['T_REC', 'HARPNUM', 'NOAA_AR', 'NOAA_ARS', 'NOAA_NUM', 'LAT_FWT', 'LON_FWT',
             'CRVAL1', 'CRVAL2', 'USFLUX', 'AREA_ACR'])
    out = config.RAW_DIR / 'NOAA_13131_2022-10-29' / 'sharp_keys.csv'
    keys.to_csv(out, index=False)
    print(f'saved -> {out}')
else:
    out = config.RAW_DIR / 'NOAA_13131_2022-10-29' / 'sharp_keys.csv'
    if out.exists():
        print(pd.read_csv(out).query('NOAA_AR == 13131').to_string(index=False))

---

## Next

`02A_data_processing.ipynb` corrects these cubes and puts them on a uniform time grid.